In [1]:
import pandas as pd
import numpy as np

In [2]:
# i want to load my file path




In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
file_path =


In [19]:
from google.colab import files
uploaded = files.upload()



Saving Education.zip to Education.zip


In [22]:
!unzip Education.zip

Streaming output truncated to the last 5000 lines.
  inflating: Education/Resume_030001_Lori_Hart.pdf  
  inflating: Education/Resume_030002_Anthony_Moses.pdf  
  inflating: Education/Resume_030003_Edward_Shepherd.pdf  
  inflating: Education/Resume_030004_Nicholas_Potter.pdf  
  inflating: Education/Resume_030005_Donna_Watts.pdf  
  inflating: Education/Resume_030006_Jennifer_Oconnor.pdf  
  inflating: Education/Resume_030007_Antonio_Barnes.pdf  
  inflating: Education/Resume_030008_Alexander_Huang.pdf  
  inflating: Education/Resume_030009_Courtney_Clark.pdf  
  inflating: Education/Resume_030010_Samuel_George.pdf  
  inflating: Education/Resume_030011_Kyle_George.pdf  
  inflating: Education/Resume_030012_Jason_Moore.pdf  
  inflating: Education/Resume_030013_Christina_Mason.pdf  
  inflating: Education/Resume_030014_Kathryn_Alexander.pdf  
  inflating: Education/Resume_030015_Nathan_White.pdf  
  inflating: Education/Resume_030016_Ethan_Carlson.pdf  
  inflating: Education/Resume_0

In [28]:
import sys
!{sys.executable} -m pip install PyPDF2

from pathlib import Path
import pandas as pd
from PyPDF2 import PdfReader


def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract text from one PDF file."""
    reader = PdfReader(str(pdf_path))

    pages_text = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    return "\n".join(pages_text).strip()


def read_pdf_folder(folder_path: str) -> pd.DataFrame:
    """Read all PDF files from a folder and return filename and text."""
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}") # Corrected f-string

    records = []

    for pdf_file in folder.glob("*.pdf"):
        try:
            records.append({
                "filename": pdf_file.name,
                "text": extract_text_from_pdf(pdf_file)
            })
        except Exception as e:
            records.append({
                "filename": pdf_file.name,
                "text": "",
                "error": str(e)
            })

    return pd.DataFrame(records)


def save_to_csv(df: pd.DataFrame, output_path: str) -> None:
    """Save DataFrame to CSV."""
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved successfully: {output_path}")


def main():
    folder_path = "/content/Education" # Updated to the unzipped Education folder
    output_path = "pdf_text_output.csv"

    df = read_pdf_folder(folder_path)

    print("Total PDFs processed:", len(df))
    print(df.head())

    save_to_csv(df, output_path)


if __name__ == "__main__":
    main()

Total PDFs processed: 5000
                             filename  \
0      Resume_034753_Dawn_Gardner.pdf   
1        Resume_034104_Brian_Mack.pdf   
2         Resume_033269_Kevin_Lin.pdf   
3  Resume_033860_Justin_Castaneda.pdf   
4   Resume_033923_Jackson_Mcguire.pdf   

                                                text  
0  Dawn Gardner\n dawn.gardner34753@hardy.info | ...  
1  Brian Mack\n brian.mack34104@perkins.info | 37...  
2  Kevin Lin\n kevin.lin33269@bradley.com | 365.7...  
3  Justin Castaneda\n justin.castaneda33860@peck....  
4  Jackson Mcguire\n jackson.mcguire33923@marshal...  
Saved successfully: pdf_text_output.csv


In [ ]:
# add folder path here


In [26]:
!pip install PyPDF2
import pandas as pd
from PyPDF2 import PdfReader

pdf_path = "/content/drive/MyDrive/20_Job_Descriptions (2).pdf"

reader = PdfReader(pdf_path)

data = []

for page in reader.pages:

    page_text = page.extract_text() or ""

    lines = [line.strip() for line in page_text.split("\n") if line.strip()]

    # First line = Job Title
    name = lines[0]

    # Remaining text = Description
    text = "\n".join(lines[1:])

    data.append({
        "name": name,
        "text": text
    })

df = pd.DataFrame(data)

print(df.shape)   # (20, 2)
print(df.head())

# Save if needed
df.to_csv("job_descriptions.csv", index=False)

(20, 2)
                               name  \
0  20 Professional Job Descriptions   
1             Azure DevOps Engineer   
2                    Data Scientist   
3                      Data Analyst   
4              Full Stack Developer   

                                                text  
0  Azure Data Engineer\nJob Summary\nWe are seeki...  
1  Job Summary\nWe are seeking a skilled Azure De...  
2  Job Summary\nWe are seeking a skilled Data Sci...  
3  Job Summary\nWe are seeking a skilled Data Ana...  
4  Job Summary\nWe are seeking a skilled Full Sta...  


In [30]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("pdf_text_output.csv") # Changed from resumes.csv

print(job_df.shape)
print(resume_df.shape)

# ==============================
# 2. Clean missing values
# ==============================

job_df["text"] = job_df["text"].fillna("")
resume_df["text"] = resume_df["text"].fillna("")

# ==============================
# 3. Load Hugging Face embedding model
# ==============================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ==============================
# 4. Generate embeddings
# ==============================

job_embeddings = model.encode(
    job_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

resume_embeddings = model.encode(
    resume_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Job Embeddings Shape:", job_embeddings.shape)
print("Resume Embeddings Shape:", resume_embeddings.shape)

# ==============================
# 5. Save embeddings as NPY files
# ==============================

np.save("job_embeddings.npy", job_embeddings)
np.save("resume_embeddings.npy", resume_embeddings)

print("Saved job_embeddings.npy")
print("Saved resume_embeddings.npy")

# ==============================
# 6. Optional: Save updated CSV also
# ==============================

job_df.to_csv("job_descriptions_clean.csv", index=False)
resume_df.to_csv("resumes_clean.csv", index=False)

print("Completed successfully")


import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("pdf_text_output.csv") # Changed from resumes.csv

# ==============================
# 2. Load NPY embeddings
# ==============================

job_embeddings = np.load("job_embeddings.npy")
resume_embeddings = np.load("resume_embeddings.npy")

print("Job Embeddings:", job_embeddings.shape)
print("Resume Embeddings:", resume_embeddings.shape)

# ==============================
# 3. Calculate Similarity / Distance
# ==============================

cosine_scores = cosine_similarity(job_embeddings, resume_embeddings)

euclidean_scores = euclidean_distances(job_embeddings, resume_embeddings)

# ==============================
# 4. Get Top 10 Resume Matches
# ==============================

results = []

for job_index in range(len(job_df)):

    job_name = job_df.loc[job_index, "name"]

    # Cosine: higher score is better
    top_cosine_indexes = np.argsort(cosine_scores[job_index])[::-1][:10]

    for rank, resume_index in enumerate(top_cosine_indexes, start=1):
        results.append({
            "job_name": job_name,
            "method": "Cosine Similarity",
            "rank": rank,
            "resume_name": resume_df.loc[resume_index, "filename"],
            "score": cosine_scores[job_index][resume_index]
        })

    # Euclidean: lower distance is better
    top_euclidean_indexes = np.argsort(euclidean_scores[job_index])[:10]

    for rank, resume_index in enumerate(top_euclidean_indexes, start=1):
        results.append({
            "job_name": job_name,
            "method": "Euclidean Distance",
            "rank": rank,
            "resume_name": resume_df.loc[resume_index, "filename"],
            "score": euclidean_scores[job_index][resume_index]
        })

# ==============================
# 5. Convert to DataFrame
# ==============================

match_df = pd.DataFrame(results)

print(match_df.head(20))

# ==============================
# 6. Save Final Output
# ==============================

match_df.to_csv("job_resume_top10_matches.csv", index=False)

print("Saved: job_resume_top10_matches.csv")

(20, 2)
(5000, 2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Job Embeddings Shape: (20, 384)
Resume Embeddings Shape: (5000, 384)
Saved job_embeddings.npy
Saved resume_embeddings.npy
Completed successfully
Job Embeddings: (20, 384)
Resume Embeddings: (5000, 384)
                            job_name              method  rank  \
0   20 Professional Job Descriptions   Cosine Similarity     1   
1   20 Professional Job Descriptions   Cosine Similarity     2   
2   20 Professional Job Descriptions   Cosine Similarity     3   
3   20 Professional Job Descriptions   Cosine Similarity     4   
4   20 Professional Job Descriptions   Cosine Similarity     5   
5   20 Professional Job Descriptions   Cosine Similarity     6   
6   20 Professional Job Descriptions   Cosine Similarity     7   
7   20 Professional Job Descriptions   Cosine Similarity     8   
8   20 Professional Job Descriptions   Cosine Similarity     9   
9   20 Professional Job Descriptions   Cosine Similarity    10   
10  20 Professional Job Descriptions  Euclidean Distance     1   
11  20

In [36]:
cosine_matches = match_df[match_df['method'] == 'Cosine Similarity'].copy()

# Create 'rank_col' for pivoting, ensuring it's a string for column names
cosine_matches['rank_col'] = 'Rank_' + cosine_matches['rank'].astype(str)

# Pivot the DataFrame to get job_name as rows and ranks as columns
final_output_df = cosine_matches.pivot(index='job_name', columns='rank_col', values='resume_name')

# Extract the numerical part of column names and sort them numerically
sorted_columns = sorted(final_output_df.columns, key=lambda x: int(x.split('_')[1]))
final_output_df = final_output_df[sorted_columns]

# Display the final DataFrame
display(final_output_df)

rank_col,Rank_1,Rank_2,Rank_3,Rank_4,Rank_5,Rank_6,Rank_7,Rank_8,Rank_9,Rank_10
job_name,,,,,,,,,,
20 Professional Job Descriptions,Resume_034496_Susan_Watson.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_033394_Steven_Watkins.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030521_Suzanne_Bowman.pdf,Resume_030174_Taylor_Johnston.pdf,Resume_032481_Christopher_Smith.pdf,Resume_034361_Brandon_Carter.pdf,Resume_034359_John_Macias.pdf,Resume_033541_Mark_Zhang.pdf
Azure DevOps Engineer,Resume_032481_Christopher_Smith.pdf,Resume_032436_Derrick_Diaz.pdf,Resume_034361_Brandon_Carter.pdf,Resume_033274_Shelly_Lewis.pdf,Resume_032194_Cindy_Ramirez.pdf,Resume_034849_Justin_Nelson.pdf,Resume_034359_John_Macias.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_030056_Christopher_Lin.pdf,Resume_030176_Amy_Horton.pdf
Big Data Engineer,Resume_033674_Edgar_Watts.pdf,Resume_034496_Susan_Watson.pdf,Resume_032076_Lisa_Nguyen.pdf,Resume_031427_Judith_King.pdf,Resume_033266_Brandon_Simon.pdf,Resume_032194_Cindy_Ramirez.pdf,Resume_034361_Brandon_Carter.pdf,Resume_032481_Christopher_Smith.pdf,Resume_033654_James_Fernandez.pdf,Resume_031306_Devin_Wallace.pdf
Business Analyst,Resume_033012_Daniel_Smith.pdf,Resume_030874_James_Soto.pdf,Resume_033461_Amanda_Harvey.pdf,Resume_032588_Robert_Jacobson.pdf,Resume_030093_David_Thompson.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_030877_Angela_Simmons.pdf,Resume_032099_Melanie_Jackson.pdf,Resume_034496_Susan_Watson.pdf,Resume_030167_Donald_Campos.pdf
Cloud Engineer,Resume_033826_Heather_Becker.pdf,Resume_030056_Christopher_Lin.pdf,Resume_032481_Christopher_Smith.pdf,Resume_031171_Christopher_Reese.pdf,Resume_032575_Kevin_Armstrong.pdf,Resume_033816_Luis_Carroll.pdf,Resume_034068_Jared_Cooley.pdf,Resume_034174_James_Salazar.pdf,Resume_032436_Derrick_Diaz.pdf,Resume_033028_Kaylee_Anderson.pdf
Cyber Security Analyst,Resume_030783_Krista_Smith.pdf,Resume_030252_Ronnie_Whitney.pdf,Resume_034629_Cody_Carter.pdf,Resume_034843_Kelli_Perez.pdf,Resume_033882_Debra_Gaines.pdf,Resume_030167_Donald_Campos.pdf,Resume_031507_Joshua_Pierce.pdf,Resume_030996_Juan_Gray.pdf,Resume_031917_Aaron_Wilson.pdf,Resume_032280_Mallory_Davis.pdf
Data Analyst,Resume_034496_Susan_Watson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_034175_Erin_Harmon.pdf,Resume_030167_Donald_Campos.pdf,Resume_034629_Cody_Carter.pdf,Resume_031929_Valerie_Martin.pdf,Resume_032602_Anthony_Abbott.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_032481_Christopher_Smith.pdf
Data Scientist,Resume_034496_Susan_Watson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_034629_Cody_Carter.pdf,Resume_031631_Todd_Burgess.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_033394_Steven_Watkins.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_030485_Alexander_Cain.pdf,Resume_032481_Christopher_Smith.pdf,Resume_034361_Brandon_Carter.pdf
Digital Marketing Specialist,Resume_032076_Lisa_Nguyen.pdf,Resume_033909_Robert_Clark.pdf,Resume_034359_John_Macias.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_033654_James_Fernandez.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030201_Brittney_Gaines.pdf,Resume_033394_Steven_Watkins.pdf,Resume_034600_Jessica_Elliott.pdf,Resume_034059_Michael_Smith.pdf
